# RAG Minecraft

#### Configuration Gemini API

In [16]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import csv
import os
import pickle
import re
import shutil
import time
import uuid
import json
from pathlib import Path
from urllib.parse import urlparse, unquote

import requests
import cloudscraper
from bs4 import BeautifulSoup
from IPython.display import Markdown

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, format_document
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.storage import LocalFileStore
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_classic.retrievers import MultiVectorRetriever

from langchain_ollama import ChatOllama

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)


/var/folders/n6/53m8gkg13qvb_3qz8npdf9lm0000gn/T/ipykernel_79100/2320296692.py:25: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [17]:
from api_config import configure_google_api_key

GOOGLE_API_KEY = configure_google_api_key()

In [18]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
)

### Config sources

In [ ]:
WIKI_PAGES =  "Minecraft"

FANDOM_URLS = [
    "https://minecraft.fandom.com/fr/wiki/Fabrication/Nourriture",
    "https://minecraft.fandom.com/fr/wiki/Enchantement",
    "https://minecraft.fandom.com/fr/wiki/Survie",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif",
    "https://minecraft.fandom.com/fr/wiki/Hardcore",
    "https://minecraft.fandom.com/fr/wiki/Fabrication",
    "https://minecraft.fandom.com/fr/wiki/Cuisson",
    "https://minecraft.fandom.com/fr/wiki/Alchimie",
    "https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures",
    "https://minecraft.fandom.com/fr/wiki/Structures",
    "https://minecraft.fandom.com/fr/wiki/Fabrication/Outils",
    "https://minecraft.fandom.com/fr/wiki/Commerce",
    "https://minecraft.fandom.com/fr/wiki/Biome",
    "https://minecraft.fandom.com/fr/wiki/Minage",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Guide_de_survie ",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Le_deuxi%C3%A8me_jour",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Les_jours_suivants",
    "https://minecraft.fandom.com/fr/wiki/Creeper ",
    "https://minecraft.fandom.com/fr/wiki/Enderman ",
    "https://minecraft.fandom.com/fr/wiki/Squelette ",
    "https://minecraft.fandom.com/fr/wiki/Zombie ",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Combat",
    "https://minecraft.fandom.com/fr/wiki/%C3%89levage",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Agriculture",
     "https://minecraft.fandom.com/fr/wiki/Chat",
     "https://minecraft.fandom.com/fr/wiki/Villageois",
     "https://minecraft.fandom.com/fr/wiki/Ender_Dragon",
     "https://minecraft.fandom.com/fr/wiki/Wither",
     "https://minecraft.fandom.com/fr/wiki/Tutoriels/Succ%C3%A8s",
     "https://minecraft.fandom.com/fr/wiki/Tutoriels/Loups",
]

#### Config llm

In [2]:
scraper = cloudscraper.create_scraper()

NameError: name 'cloudscraper' is not defined

In [25]:
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

summarize_chain = (
    PromptTemplate.from_template(
        """
Crée un résumé optimisé pour la recherche sémantique.

Conserve :
- les noms d'objets
- les noms de créatures
- les noms de structures
- les noms de biomes
- les mécaniques

N'invente rien.
Ne fais aucune introduction.
Ne fais aucune conclusion.

Texte :

{doc}

Résumé :
"""
    )
    | llm
    | StrOutputParser()
)

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)

## Utils

### General

In [3]:
from pathlib import Path


def get_all_relative_filenames(directory_path: str = "files/") -> list[str]:
    """Retourne les chemins relatifs de tous les fichiers (sans l'extension .txt)

    en conservant la structure des sous-dossiers.
    """
    root = Path(directory_path)

    paths = []
    for f in root.rglob("*"):
        if f.is_file():
            # 1. On obtient le chemin relatif par rapport au dossier racine
            #    Ex: "files/jeux/zelda.txt" -> "jeux/zelda.txt"
            relative_path = f.relative_to(root)

            # 2. On retire l'extension .txt (ou autre) en prenant le parent + le stem
            #    Ex: "jeux/zelda.txt" -> "jeux/zelda"
            if relative_path.parent != Path("."):
                # Si le fichier est dans un sous-dossier
                clean_path = f"{relative_path.parent}/{relative_path.stem}"
            else:
                # Si le fichier est directement à la racine de "files/"
                clean_path = relative_path.stem

            paths.append(clean_path)

    return paths

In [4]:
def page_loaded(page_name: str) -> bool:
    file_name = Path(f"files/{page_name}.txt")
    return file_name.exists()

In [5]:
def write_txt(file_name, paragraphs):
    # Crée le dossier parent automatiquement s'il n'existe pas
    folder = os.path.dirname(file_name)
    if folder:
        os.makedirs(folder, exist_ok=True)

    # Sauvegarde au format JSON Lines (JSONL) pour éviter les corruptions de texte
    with open(file_name, "w", encoding="utf-8") as f:
        for p in paragraphs:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

In [ ]:
from pathlib import Path


def get_all_filenames(directory_path: str = "files/") -> list[str]:
    """Retourne la liste de tous les noms de fichiers dans le dossier spécifié,

    y compris dans les sous-dossiers.
    """
    path = Path(directory_path)

    # .rglob("*") cherche absolument tout (fichiers et dossiers) récursivement.
    # On filtre avec f.is_file() pour ne garder que les fichiers.
    return [f.name for f in path.rglob("*") if f.is_file() and f != "wikipedia.txt"]

In [7]:
import shutil
import os

def reset_vector_db(
    chroma_dir="./chroma_minecraft_multivec",
    store_dir="./local_chunks_store"
):
    """
    Supprime complètement Chroma et le LocalFileStore.
    """

    # Fermer les objets si déjà chargés
    try:
        vectorstore._client.reset()
    except:
        pass

    # Supprimer Chroma
    if os.path.exists(chroma_dir):
        shutil.rmtree(chroma_dir)
        print(f"Supprimé : {chroma_dir}")

    # Supprimer les chunks bruts
    if os.path.exists(store_dir):
        shutil.rmtree(store_dir)
        print(f"Supprimé : {store_dir}")

    # Recréer dossiers vides
    os.makedirs(chroma_dir, exist_ok=True)
    os.makedirs(store_dir, exist_ok=True)

    print("Base réinitialisée.")

#### Wikipedia

In [8]:
def parse_wikipedia_sections(soup, source):

    docs = []

    h2 = None
    h3 = None
    h4 = None

    current_text = []

    def save_section():
        nonlocal current_text

        text = "\n".join(current_text).strip()

        # ignorer sections vides
        if not text:
            current_text = []
            return

        title_parts = [x for x in [h2, h3, h4] if x]
        section_title = " > ".join(title_parts)
        if section_title in ["", " "]:
            section_title_title = "Introduction"

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": source,
                    "section": section_title
                }
            )
        )

        current_text = []

    for tag in soup.find_all(["h2", "h3", "h4", "p", "li"]):

        txt = tag.get_text(" ", strip=True)

        if not txt:
            continue

        # arrêt avant les sections inutiles
        if tag.name == "h2" and any(
            x in txt
            for x in [
                "Références",
                "Notes et références",
                "Voir aussi",
                "Liens externes",
                "Accueil",
                "Autres versions"
            ]
        ):
            break

        if tag.name == "h3" and any(
            x in txt
            for x in [
                "Autres versions"
            ]
        ):
            break

        if tag.name == "h2":

            save_section()

            h2 = txt
            h3 = None
            h4 = None

        elif tag.name == "h3":

            save_section()

            h3 = txt
            h4 = None

        elif tag.name == "h4":

            save_section()

            h4 = txt

        else:
            current_text.append(txt)

    save_section()

    return docs

#### Fandom

In [9]:
def clean_wiki_text(text: str) -> str:

    # Guillemets typographiques → guillemets simples
    text = text.replace("\u00ab", '"').replace("\u00bb", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2026", "...")

    # Liens wiki [[Page|texte]] → texte, [[Page]] → Page
    text = re.sub(r"\[\[(?:[^|\]]*\|)?([^\]]+)\]\]", r"\1", text)

    # Templates simples {{nom|val}} → val, {{nom}} → supprimé
    text = re.sub(r"\{\{[^}]*\|([^}]*)\}\}", r"\1", text)
    text = re.sub(r"\{\{[^}]*\}\}", "", text)

    # Balises HTML résiduelles
    text = re.sub(r"<[^>]+>", "", text)

    # Références [1], [note 2], etc.
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\[note \d+\]", "", text)

    # Marqueurs de version [Version Java 1.x]
    text = re.sub(r"\[Version [^\]]+\]", "", text)

    # Lignes qui ne contiennent que de la ponctuation ou des caractères spéciaux
    text = re.sub(r"^[^\w\s]{1,5}$", "", text, flags=re.MULTILINE)

    # Espaces multiples sur une même ligne
    text = re.sub(r"[ \t]+", " ", text)

    # Lignes vides multiples → une seule
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

def remove_empty_sections(text: str) -> str:
    """
    Supprime les titres de section qui ne sont pas suivis de contenu.
    Ex : 'SECTION: Galerie\n\nSECTION: Combat\n' → garde seulement Combat s'il a du contenu.
    """
    lines = text.split("\n")
    result = []
    i = 0

    while i < len(lines):
        line = lines[i]

        if line.startswith("SECTION:") or line.startswith("SOUS-SECTION:"):
            # Chercher si la section a du contenu
            j = i + 1
            # Sauter les lignes vides juste après le titre
            while j < len(lines) and lines[j].strip() == "":
                j += 1

            # S'il y a du contenu non-vide et que ce n'est pas une autre section
            if j < len(lines) and not (
                lines[j].startswith("SECTION:") or lines[j].startswith("SOUS-SECTION:")
            ):
                result.append(line)
            # Sinon on saute le titre de section vide
        else:
            result.append(line)

        i += 1

    return "\n".join(result)

def remove_noise_lines(text: str) -> str:
    """
    Supprime les lignes qui ne contiennent pas assez d'information :
    - moins de 20 caractères sauf si c'est un titre de section
    - lignes qui ressemblent à des noms de fichier (image, .png, .jpg)
    - lignes qui ne sont que des chiffres ou caractères spéciaux
    """
    lines = text.split("\n")
    result = []

    for line in lines:
        stripped = line.strip()

        # Toujours garder les titres de section
        if stripped.startswith("SECTION:") or stripped.startswith("SOUS-SECTION:"):
            result.append(line)
            continue

        # Supprimer les noms de fichiers image
        if re.match(r"(?i).*\.(png|jpg|jpeg|gif|svg|webp|ogg|mp3|wav)$", stripped):
            continue

        # Supprimer les lignes trop courtes (bruit)
        if len(stripped) < 20 and stripped != "":
            continue

        # Supprimer les lignes qui ne sont que des chiffres/ponctuation
        if re.match(r"^[\d\s.,;:!?()\[\]«»\"'/-]+$", stripped):
            continue

        if stripped.startswith("v d m"):
            continue

        if stripped.startswith("↑"):
            continue

        if "google.fr/search" in stripped:
            continue

        if "Version Bedrock" in stripped and len(stripped) > 100:
            continue

        if "Historique des versions" in stripped and len(stripped) > 100:
            continue

        result.append(line)

    return "\n".join(result)

STOP_HEADINGS = {
    "références", "notes et références", "notes diverses",
    "voir aussi", "liens externes", "annexes",
    "galerie", "historique", "succès",
    "créatures des autres jeux", "créatures de minecraft earth",
    "créatures de minecraft dungeons", "créatures prévues",
    "créatures inutilisées", "créatures supprimées",
    "créatures non implémentées", "créatures de la version éducation",
    "tutoriels","sons",
}

def extract_list(tag) -> list:
    items = []
    for li in tag.find_all("li", recursive=False):
        text_parts = []
        for node in li.children:
            if not hasattr(node, "name"):
                text_parts.append(str(node))
            elif node.name not in ["ul", "ol"]:
                text_parts.append(node.get_text(" ", strip=True))
        text = clean_wiki_text(" ".join(text_parts))
        if text and len(text) >= 15:
            items.append(f"- {text}")
        for sub in li.find_all(["ul", "ol"], recursive=False):
            for sub_item in extract_list(sub):
                items.append(f"  {sub_item}")
    return items

def get_cell_name(td):
    td = BeautifulSoup(str(td), "html.parser")

    # Remplacer les liens par leur title
    for a in td.find_all("a"):
        title = a.get("title", "").strip()
        if title and "modifier" not in title.lower():
            a.replace_with(title)

    # Remplacer les images par leur alt
    for img in td.find_all("img"):
        alt = img.get("alt", "").strip()
        if alt:
            img.replace_with(alt)

    return td.get_text(" ", strip=True)

def extract_table(tag) -> list:
    rows = []

    # Récupérer les en-têtes
    headers = []
    header_row = tag.find("tr")
    if header_row:
        headers = [
            clean_wiki_text(th.get_text(" ", strip=True))
            for th in header_row.find_all("th")
            if len(th.get_text(strip=True)) >= 2
        ]

    for tr in tag.find_all("tr"):
        cells = []
        for td in tr.find_all("td"):
            cell_text = clean_wiki_text(get_cell_name(td))
            if cell_text and len(cell_text) >= 2:
                cells.append(cell_text)

        if not cells:
            continue

        if headers and len(headers) == len(cells):
            parts = [f"{h} : {c}" for h, c in zip(headers, cells)]
            rows.append("- " + " | ".join(parts))
        else:
            rows.append("- " + " : ".join(cells))

    return rows


### Scraping functions

#### Wikipedia

In [10]:
def scrape_wikipedia(title: str):

    url = "https://fr.wikipedia.org/w/api.php"

    params = {
        "action": "parse",
        "page": title,
        "format": "json",
        "prop": "text",
        "redirects": True
    }

    r = requests.get(
        url,
        params=params,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    data = r.json()

    html = data["parse"]["text"]["*"]
    soup = BeautifulSoup(html, "html.parser")
    docs = parse_wikipedia_sections(soup, f"wikipedia:{title}" )
    
    docs = [
        {
            "page_content": doc.page_content,
            "source": doc.metadata.get("source"),
            "section": doc.metadata.get("section")
        } 
        for doc in docs
    ]

    write_txt(
        f"files/wikipedia.txt",
        docs
    )
    return docs


#### Fandom

In [29]:
def scrape_fandom(url: str, page_name = str, mode:str ="windows"):

    if mode == "mac":

        r = requests.get(
           url,
           headers={"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"},
           timeout=30,
       )
                

    else:

        headers_req = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0 Safari/537.36"
            ),
            "Accept-Language": "fr-FR,fr;q=0.9",
            "Referer": "https://www.google.com/"
        }

        r = scraper.get(url, headers=headers_req)

    soup = BeautifulSoup(r.text, "html.parser")

    content = soup.find("div", {"class": "mw-parser-output"})
    if not content:
        return []

    # ==========================
    # CLEAN STATIC BLOCKS
    # ==========================
    selectors_to_remove = [
        "#toc",
        ".mw-references-wrap",
        "ol.references",
        ".navbox",
        ".vertical-navbox",
        ".portable-infobox",
        ".catlinks",
        ".mw-editsection",
        ".reference"
    ]

    for selector in selectors_to_remove:
        for element in content.select(selector):
            element.decompose()

    docs = []

    h2 = None
    h3 = None
    h4 = None

    current_content = []

    def save_section():
        nonlocal current_content

        text = "\n".join(current_content).strip()

        text = remove_empty_sections(text)
        text = remove_noise_lines(text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()

        if not text:
            current_content = []
            return

        section_parts = [x for x in [h2, h3, h4] if x]

        section_title = (
            "Introduction"
            if not section_parts
            else " > ".join(section_parts)
        )

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": url,
                    "section": section_title
                }
            )
        )

        current_content = []

    # ==========================
    # MAIN LOOP
    # ==========================
    for tag in content.children:

        if not getattr(tag, "name", None):
            continue

        # ==========================
        # HEADINGS — h1 inclus pour FAQ
        # ==========================
        heading_tag = None

        if tag.name in ["h1", "h2", "h3", "h4"]:
            heading_tag = tag
        elif tag.name == "div" and "mw-heading" in tag.get("class", []):
            heading_tag = tag.find(["h1", "h2", "h3", "h4"], recursive=False)

        if heading_tag:
            raw_heading = heading_tag.get_text(" ", strip=True)
            heading = clean_wiki_text(raw_heading)
            heading = re.sub(r"\[.*?\]", "", heading).strip()

            if not heading:
                continue

            # ==========================
            # STOP SECTIONS
            # ==========================
            if heading.lower() in STOP_HEADINGS:
                save_section()
                break

            # ==========================
            # FAQ DETECTION
            # ==========================
            b_texts = [b.get_text(strip=True).lower() for b in heading_tag.find_all("b")]
            is_faq = any(t == "q:" for t in b_texts)

            if is_faq:
                # Sauvegarder la section précédente (Q/R précédente)
                save_section()

                # Extraire le texte de la question sans le préfixe "Q:"
                question_text = clean_wiki_text(heading)
                question_text = re.sub(r"^Q\s*:\s*", "", question_text, flags=re.IGNORECASE).strip()

                # Chaque question devient sa propre section
                h2 = f"FAQ > {question_text}"
                h3 = None
                h4 = None

                # Ajouter la question en début de contenu
                current_content.append(f"Q: {question_text}")
                continue

            # ==========================
            # SECTION NORMALE
            # ==========================
            save_section()

            if heading_tag.name in ["h1", "h2"]:
                h2 = heading
                h3 = None
                h4 = None
            elif heading_tag.name == "h3":
                h3 = heading
                h4 = None
            elif heading_tag.name == "h4":
                h4 = heading

            continue

        # ==========================
        # PARAGRAPHS
        # ==========================
        if tag.name == "p":
            text = clean_wiki_text(tag.get_text(" ", strip=True))
            if not text or len(text) < 10:
                continue
            current_content.append(text)

        # ==========================
        # DL/DD — réponses FAQ et définitions
        # ==========================
        elif tag.name == "dl":
            for dd in tag.find_all("dd", recursive=True):
                raw = dd.get_text(" ", strip=True)
                # Supprimer le préfixe "R:" s'il existe
                raw = re.sub(r"^R\s*:\s*", "", raw, flags=re.IGNORECASE).strip()
                text = clean_wiki_text(raw)
                if not text or len(text) < 10:
                    continue
                current_content.append(text)

        # ==========================
        # LISTS
        # ==========================
        elif tag.name in ["ul", "ol"]:
            current_content.extend(extract_list(tag))

        # ==========================
        # TABLES
        # ==========================
        elif tag.name == "table":
            current_content.extend(extract_table(tag))

    save_section()

    page_name = unquote(url.split("/wiki/")[-1])

    print(f"FANDOM OK: {page_name} ({len(docs)} sections)")

    docs = [
        {
            "page_content": doc.page_content,
            "source": doc.metadata.get("source"),
            "section": doc.metadata.get("section")
        }
        for doc in docs
    ]

    write_txt(f"files/{page_name}.txt", docs)

    return docs

### Build dataset functions

In [12]:
def clean_database():
    dossiers_a_nettoyer = [
        "files",        # Tes pages brutes en JSONL
        "chroma_minecraft_multivec",    # La base de données Chroma
        "local_store"   # Le dossier de ton LocalFileStore (parfois nommé 'docstore' ou autre)
    ]

    print("=== NETTOYAGE EN COURS ===")
    for dossier in dossiers_a_nettoyer:
        if os.path.exists(dossier):
            try:
                shutil.rmtree(dossier)
                print(f"Dossier '{dossier}' supprimé avec succès.")
            except Exception as e:
                print(f"Erreur lors de la suppression de '{dossier}' : {e}")
                print("Pense bien à faire 'Kernel -> Restart' avant de lancer cette cellule.")
        else:
            print(f"Le dossier '{dossier}' est déjà absent (propre).")

In [13]:
def scrape_web(platform : str = "windows"):

    new_pages = []

    # if not page_loaded("wikipedia"):
    #     try:
    #         scrape_wikipedia(WIKI_PAGES)
    #         print("Chargement de wikipedia OK")
    #         new_pages.append("wikipedia")
    #     except Exception as e:
    #         print("Soucis avec le chargement de la page wikipedia:", e)
    # else:
    #     print("Wikipedia deja chargee")
    
    for url in FANDOM_URLS:
        
        parsed = urlparse(url)

        page_name = unquote(
            parsed.path.split("/wiki/")[-1]
        )     

        if not page_loaded(page_name):
            
            try:

                doc = scrape_fandom(url, page_name, mode=platform)

                if doc:
                    new_pages.append(page_name)
                    print(f"Chargement de la page {page_name} OK")

            except Exception as e:
                print("Soucis avec le chargement de la page wiki:", url, e)
        
        else:
            print(f"Wiki : {page_name} deja chargee")

    return new_pages

In [14]:
def load_txt_documents(page_names: list[str]):
    documents = []
    
    for page in page_names:
        file_path = f"files/{page}.txt"
        print(f"Lecture du fichier : {file_path}")
        
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    
                    # Lecture directe du JSON de la ligne
                    data = json.loads(line)
                    
                    metadata = {
                        "source": data.get("source", ""),
                        "section": data.get("section", ""),
                        "file_origin": file_path
                    }
                    
                    doc = Document(
                        page_content=data.get("page_content", ""), 
                        metadata=metadata
                    )
                    documents.append(doc)
                            
        except Exception as e:
            print(f"Impossible de lire le fichier {file_path} : {e}")
                    
    print(f"\n Chargement terminé ! Total de {len(documents)} Documents récupérés.")
    return documents

## Load document and chunks

In [17]:
# reset_vector_db()

Supprimé : ./chroma_minecraft_multivec
Supprimé : ./local_chunks_store
Base réinitialisée.


In [123]:
# writes pages that have not been loaded yet as txt, and returns the new pages names
new_pages = scrape_web(platform="mac")

Wiki : Chat deja chargee
Wiki : Villageois deja chargee
Wiki : Ender_Dragon deja chargee
Wiki : Wither deja chargee
Wiki : Tutoriels/Succès deja chargee
Wiki : Tutoriels/Loups deja chargee


In [124]:

new_pages = get_all_relative_filenames()
print(len(new_pages))
print(new_pages)
# reads the new pages content as Documents


34
['Biome', 'Enderman ', 'Créatures', 'Creeper ', 'Créatif', 'La_Foire_aux_Questions', 'Fabrication', 'Wither', 'Élevage', 'Hardcore', 'Ender_Dragon', 'Alchimie', 'Survie', 'Zombie ', 'Chat', 'Commerce', 'Structures', 'Enchantement', 'Villageois', 'Minage', 'Squelette ', 'Cuisson', 'Fabrication/Nourriture', 'Fabrication/Outils', 'Tutoriels/Loups', 'Tutoriels/Combat', 'Tutoriels/Les_jours_suivants', 'Tutoriels/Agriculture', 'Tutoriels/Succès', "Tutoriels/L'End_et_l'Ender_Dragon", 'Tutoriels/Choses_à_ne_PAS_faire', 'Tutoriels/Survie_dans_le_Nether', 'Tutoriels/Guide_de_survie ', 'Tutoriels/Le_deuxième_jour']


In [ ]:
all_docs = load_txt_documents(new_pages)

### Chunking

In [126]:
def split_docs(all_docs):
    # Sécurité : Si la liste est vide, on s'arrête gentiment ici sans crasher
    if not all_docs:
        print("ANCIENS DOCS (Paragraphes bruts) : 0")
        print("NOUVEAUX CHUNKS ÉQUILIBRÉS : 0")
        print("Aucun nouveau document à découper (Tout est déjà à jour).")
        return []

    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1800, 
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    split_docs = splitter.split_documents(all_docs)

    print("ANCIENS DOCS (Paragraphes bruts) :", len(all_docs))
    print("NOUVEAUX CHUNKS ÉQUILIBRÉS :", len(split_docs))

    tailles = [len(doc.page_content) for doc in split_docs]
    if tailles:
        print(f"Taille Min : {min(tailles)} | Taille Max : {max(tailles)} | Moyenne : {int(sum(tailles)/len(tailles))}")
    
    return split_docs

In [127]:
all_docs = split_docs(all_docs)

ANCIENS DOCS (Paragraphes bruts) : 481
NOUVEAUX CHUNKS ÉQUILIBRÉS : 608
Taille Min : 50 | Taille Max : 1799 | Moyenne : 886


## Summarize + Initialize embedding model

In [137]:
## ==========================================
## MULTI-VECTOR STORAGE (CHROMA + DISK STORE)
## ==========================================

import os
import uuid
import time # Importé pour le sleep du retry

# 1. Définition des chemins sur le disque
CHROMA_DIR = "./db/chroma_minecraft_multivec"
STORE_DIR = "./db/local_chunks_store"

# 2. Initialisation de la base vectorielle Chroma (Contient les résumés)
vectorstore = Chroma(
    collection_name="minecraft_summaries",
    persist_directory=CHROMA_DIR,
    embedding_function=gemini_embeddings
)

# 3. Initialisation du stockage persistant sur DISQUE (Contient les chunks bruts)
fs_store = LocalFileStore(STORE_DIR)

# 4. Création du Retriever Multi-Vecteurs
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=fs_store,
    id_key="doc_id",
)

In [ ]:


# ==========================================
# LOGIQUE D'INDEXATION ÉCONOMIQUE (ANTI-DUPLICATA)
# ==========================================

try:
    existing_data = vectorstore.get()

    existing_sections = set(
        f"{meta.get('source')}_{meta.get('section')}"
        for meta in existing_data.get("metadatas", [])
        if meta
    )

    print(
        f"Base de données trouvée : "
        f"{len(existing_sections)} sections uniques déjà présentes dans Chroma."
    )

except Exception:
    existing_sections = set()
    print(
        "Base vectorielle vierge ou inaccessible. "
        "Première indexation."
    )

docs_to_index = [
    doc
    for doc in all_docs
    if f"{doc.metadata.get('source')}_{doc.metadata.get('section')}"
    not in existing_sections
]

if not docs_to_index:
    print(
        "Tous les documents sont déjà à jour sur le disque. "
        "ZÉRO crédit consommé !"
    )

else:
    print(
        f"{len(docs_to_index)} nouveaux chunks à traiter "
        f"(Génération Ollama + Embedding Gemini)..."
    )

    for i, doc in enumerate(docs_to_index):
        text_len = len(doc.page_content)

        print(
            f"[{i+1}/{len(docs_to_index)}] "
            f"Analyse du chunk ({text_len} caractères)...",
            end=""
        )

        # ==========================================
        # RÉSUMÉ OLLAMA
        # ==========================================
        if text_len < 700:
            summary_text = doc.page_content
        else:
            try:
                summary_text = summarize_chain.invoke(
                    {"doc": doc.page_content}
                ).strip()
            except Exception as e:
                print(f"\nErreur Ollama sur le chunk {i}: {e}")
                summary_text = doc.page_content

        # ==========================================
        # ENRICHISSEMENT PAGE + SECTION
        # ==========================================
        section = doc.metadata.get("section", "")
        file_origin = doc.metadata.get("file_origin", "")
        page_name = os.path.basename(file_origin)
        page_name = os.path.splitext(page_name)[0]

        summary_text = f"""
Page : {page_name}

Section : {section}

Résumé :
{summary_text}
""".strip()

        # ==========================================
        # ID UNIQUE
        # ==========================================
        chunk_id = str(uuid.uuid4())
        summary_doc = Document(
            page_content=summary_text,
            metadata={
                **doc.metadata,
                "doc_id": chunk_id
            }
        )

        # ==========================================
        # SAUVEGARDE AVEC RETRY AUTOMATIQUE POUR GEMINI
        # ==========================================
        max_gemini_retries = 5
        saved_successfully = False

        for gemini_attempt in range(max_gemini_retries):
            try:
                # Étape 1 : Appel API Gemini Embedding + Ajout Chroma
                retriever.vectorstore.add_documents([summary_doc])
                
                # Étape 2 : Sauvegarde locale du chunk brut
                retriever.docstore.mset([(chunk_id, doc)])
                
                print(" -> Sauvegardé !")
                saved_successfully = True
                break # On sort de la boucle de retry interne si tout est OK
                
            except Exception as e:
                # Si c'est un problème d'API (comme la 503), on attend et on réessaie
                if gemini_attempt < max_gemini_retries - 1:
                    wait_time = 5 * (gemini_attempt + 1)
                    print(f"\n⚠️ [Gemini 503] Erreur d'embedding au chunk {i}. Réessai dans {wait_time}s... (Erreur : {e})")
                    time.sleep(wait_time)
                else:
                    print(f"\n❌ Erreur persistante après {max_gemini_retries} essais sur le chunk {i} : {e}")

        # Si après les retries ça n'a pas fonctionné, on arrête proprement le script complet
        if not saved_successfully:
            print("Arrêt de sécurité. Relance la cellule plus tard quand l'API Gemini sera stable.")
            break

print("\n--- STATISTIQUES ---")
print("Total Chunks générés ce run (split_docs):", len(all_docs))
print("Total Résumés actuellement dans Chroma:", vectorstore._collection.count())

#### Test

In [133]:
from pathlib import Path

# Définis tes deux chemins ici
dossier_A = Path("./db/local_chunks_store")
dossier_B = Path("./local_chunks_store")

# Récupérer les identifiants (noms de fichiers) de chaque dossier
chunks_A = set(f.name for f in dossier_A.glob("*") if f.is_file())
chunks_B = set(f.name for f in dossier_B.glob("*") if f.is_file())

# Calculs d'intersections (mathématiques des ensembles)
doublons = chunks_A.intersection(chunks_B)
uniques_A = chunks_A - chunks_B
uniques_B = chunks_B - chunks_A

print("=== BILAN DE L'ANALYSE ===")
print(f"Dossier A (dans db) : {len(chunks_A)} éléments")
print(f"Dossier B (dehors)  : {len(chunks_B)} éléments")
print("-" * 30)
print(f"🔄 Doublons parfaits (présents dans les deux) : {len(doublons)}")
print(f"📦 Uniques au Dossier A : {len(uniques_A)}")
print(f"📦 Uniques au Dossier B : {len(uniques_B)}")
print(f"✨ Total théorique après fusion : {len(chunks_A.union(chunks_B))} chunks uniques.")

=== BILAN DE L'ANALYSE ===
Dossier A (dans db) : 633 éléments
Dossier B (dehors)  : 285 éléments
------------------------------
🔄 Doublons parfaits (présents dans les deux) : 285
📦 Uniques au Dossier A : 348
📦 Uniques au Dossier B : 0
✨ Total théorique après fusion : 633 chunks uniques.


In [157]:
def analyser_et_collecter_problemes(all_docs, vectorstore):
    """Analyse les tailles et retourne la liste des doc_ids

    dont le résumé est plus grand que le chunk original.
    """
    # Liste pour stocker les vrais doc_id problématiques
    liste_id_problemes = []

    print("Extraction des données de Chroma en cours...")
    chroma_data = vectorstore.get()
    
    # Cartographie de Chroma { "source_section": {"text": ..., "doc_id": ...} }
    resumes_map = {}
    metadatas = chroma_data.get("metadatas", [])
    documents = chroma_data.get("documents", [])
    
    for meta, doc_text in zip(metadatas, documents):
        if meta and "source" in meta and "section" in meta:
            cle_unique = f"{meta['source']}_{meta['section']}"
            resumes_map[cle_unique] = {
                "text": doc_text,
                "doc_id": meta.get("doc_id") # On attrape le vrai UUID ici
            }

    print("=== ANALYSE ET DECTION DES ANOMALIES ===")
    print(f"{'ID Boucle':<10} | {'Taille Origine':<14} | {'Taille Résumé':<13} | {'Statut'}")
    print("-" * 65)

    for i, doc in enumerate(all_docs):
        chunk_text = doc.page_content
        len_origine = len(chunk_text)
        cle_chunk = f"{doc.metadata.get('source')}_{doc.metadata.get('section')}"

        if len_origine < 700:
            continue
            
        if cle_chunk in resumes_map:
            summary_text = resumes_map[cle_chunk]["text"]
            vrai_id = resumes_map[cle_chunk]["doc_id"]
        else:
            continue

        len_resume = len(summary_text)

        if len_resume > len_origine:
            pct_reduction = ((len_origine - len_resume) / len_origine) * 100
            # Bingo ! On ajoute le vrai ID à notre liste de surveillance
            if vrai_id:
                liste_id_problemes.append(vrai_id)
                
            print(f"{i+1:<10} | {len_origine:<14} | {len_resume:<13} | ❌ PLUS GRAND (+{abs(pct_reduction):.1f}%)")

    print(f"\nAnalyse terminée. {len(liste_id_problemes)} anomalies détectées.")
    return liste_id_problemes

# Pour l'exécuter, passe-lui directement ton vectorstore Chroma :
liste_id = analyser_et_collecter_problemes(all_docs, vectorstore)

Extraction des données de Chroma en cours...
=== ANALYSE ET DECTION DES ANOMALIES ===
ID Boucle  | Taille Origine | Taille Résumé | Statut
-----------------------------------------------------------------
110        | 1019           | 1065          | ❌ PLUS GRAND (+4.5%)
112        | 1185           | 1616          | ❌ PLUS GRAND (+36.4%)
113        | 1568           | 1616          | ❌ PLUS GRAND (+3.1%)
114        | 1036           | 1099          | ❌ PLUS GRAND (+6.1%)
115        | 899            | 965           | ❌ PLUS GRAND (+7.3%)
116        | 1155           | 1206          | ❌ PLUS GRAND (+4.4%)
161        | 897            | 933           | ❌ PLUS GRAND (+4.0%)
225        | 1484           | 1502          | ❌ PLUS GRAND (+1.2%)
268        | 1792           | 1877          | ❌ PLUS GRAND (+4.7%)
269        | 1796           | 1877          | ❌ PLUS GRAND (+4.5%)
270        | 1629           | 1877          | ❌ PLUS GRAND (+15.2%)
275        | 937            | 14395         | ❌ PLUS GRA

In [160]:
def afficher_resume_par_id(vrai_id, vectorstore):
    """Cherche et affiche le résumé complet présent dans Chroma

    correspondant au doc_id (UUID) fourni.
    """
    chroma_data = vectorstore.get()
    metadatas = chroma_data.get("metadatas", [])
    documents = chroma_data.get("documents", [])
    
    # Recherche linéaire dans l'extraction Chroma
    found = False
    for meta, doc_text in zip(metadatas, documents):
        if meta and meta.get("doc_id") == vrai_id:
            found = True
            print("=" * 60)
            print(f"🆔 VRAI ID SÉLECTIONNÉ : {vrai_id}")
            print(f"📄 PAGE d'origine      : {meta.get('file_origin', 'Inconnue')}")
            print(f"📂 SECTION d'origine   : {meta.get('section', 'Inconnue')}")
            print("-" * 60)
            print("📝 CONTENU DU RÉSUMÉ STOCKE :")
            print(doc_text)
            print("=" * 60)
            break
            
    if not found:
        print(f"❌ Impossible de trouver le doc_id '{vrai_id}' dans la base Chroma actuelle.")


In [163]:
print(liste_id)

['3d6c2f7e-c17e-4713-bdb6-4400eaaa71a3', 'ceb6e8ee-6c79-4e39-a21d-11c02f8a0999', 'ceb6e8ee-6c79-4e39-a21d-11c02f8a0999', 'f6918718-ee8c-46d2-8ed9-62a39fd18453', '24703873-0a7f-4195-8f54-90505fd7dece', 'b7a4a70f-ce38-4ab8-8378-e4b74736857d', '65fd89e7-ff80-4235-9183-76ac987cec6c', 'b3b980c3-7dc5-45b3-af54-5c0a117ea86f', 'bf85ab7d-93b5-4040-b2e8-293d7e7b2cd0', 'bf85ab7d-93b5-4040-b2e8-293d7e7b2cd0', 'bf85ab7d-93b5-4040-b2e8-293d7e7b2cd0', 'e30df07e-1b71-4103-9560-6498e36cfc8c', '0ac5c760-a734-4b94-b77b-e946716adcb1', '13674463-d87c-47e8-8464-14dada44eb28', '183d720f-6087-4b55-beff-0758bef87bc3', '1fab18c5-e369-4fcc-82c9-8a64945af389', 'c19aab57-4543-406b-9f0d-6ebc6861d056']


In [162]:


# Exemple : Si la liste n'est pas vide, on inspecte le tout premier élément de la liste
for id in liste_id:
    afficher_resume_par_id(id, vectorstore)
else:
    print("Félicitations, aucun résumé n'a dépassé son chunk !")

🆔 VRAI ID SÉLECTIONNÉ : 3d6c2f7e-c17e-4713-bdb6-4400eaaa71a3
📄 PAGE d'origine      : files/Wither.txt
📂 SECTION d'origine   : Apparition
------------------------------------------------------------
📝 CONTENU DU RÉSUMÉ STOCKE :
Page : Wither

Section : Apparition

Résumé :
Le Wither peut être invoqué en plaçant quatre blocs de sable des âmes ou de terre des âmes en forme de T (comme dans le patron ci-contre), puis en plaçant trois crânes de Wither squelette sur les trois blocs supérieurs. Le dernier bloc placé doit être un des trois crânes de Wither squelette . Des blocs d' air sont nécessaires dans la partie inférieure de la structure, sans quoi le Wither n'apparaîtra pas. Les crânes peuvent être placés avec des distributeurs , ce qui permet l'automatisation du processus. La structure peut être construite dans n'importe quel sens, à l'envers comme à l'horizontale. Le Wither apparaîtra au pied de la structure.
Tenter de faire apparaître un Wither en difficulté Paisible ne fera pas trans

In [140]:
# Affiche le résumé du chunk 290 (index 289 dans la liste)
print(vectorstore.get()["documents"][289])

Page : Minage

Section : Dureté des blocs

Résumé :
Hache : 
- 1,5 : 0,75 : 0,4 : 0,25 : 0,2 : 0,2 : 0,15
- 1,5 : 0,75 : 0,4 : 0,25 : 0,2 : 0,2 : 0,15
- 1,5 : 0,75 : 0,4 : 0,25 : 0,2 : 0,2 : 0,15
- 1,5 : 0,75 : 0,4 : 0,25 : 0,2 : 0,2 : 0,15
- 1,5 : 0,75 : 0,4 : 0,25 : 0,2 : 0,2 : 0,15
- 1,25 : 0,95 : 0,5 : 0,35 : 0,25 : 0,25 : 0,2

Pioche :
- 1,5 : 7,5 : 1,15 : 0,6 : 0,4 : 0,3 : 0,25 : 0,2
- 1,5 : 7,5 : 1,15 : 0,6 : 0,4 : 0,3 : 0,25 : 0,2
- 1,5 : 7,5 : 1,15 : 0,6 : 0,4 : 0,3 : 0,25 : 0,2
- 1,5 : 7,5 : 1,15 : 0,6 : 0,4 : 0,3 : 0,25 : 0,2
- 1,5 : 7,5 : 1,15 : 0,6 : 0,4 : 0,3 : 0,25 : 0,2
- 1,25 : 6,25 : 0,95 : 0,5 : 0,35 : 0,25 : 0,25 : 0,2

Pelle :
- 0,6 : 0,9 : 0,45 : 0,25 : 0,15 : 0,15 : 0,1 : 0,1
- 0,6 : 0,9 : 0,45 : 0,25 : 0,15 : 0,15 : 0,1 : 0,1
- 0,6 : 0,9 : 0,45 : 0,25 : 0,15 : 0,15 : 0,1 : 0,1
- 0,6 : 0,9 : 0,45 : 0,25 : 0,15 : 0,15 : 0,1 : 0,1
- 0,6 : 0,9 : 0,45 : 0,25 : 0,15 : 0,15 : 0,1 : 0,1


In [154]:
print(len(vectorstore.get()["documents"][290]))
print(len(vectorstore.get()["documents"][335]))

1502
148


In [34]:
# data = vectorstore.get()

# print("Nb docs :", len(data["documents"]))



# for i, (doc, meta) in enumerate(zip(data["documents"], data["metadatas"])):
#     print(f"\n--- DOC {i} ---")
#     print("META:", meta)
#     print("TEXT:", doc[:1000])

# print("Résumés :", len(data["documents"]))
# print("Métadatas :", len(data["metadatas"]))

# print(data.keys())

# results = vectorstore.get()


results = vectorstore.get(
    where={"file_origin": "files/Fabrication/Nourriture.txt"},
    include=["documents", "metadatas"]
)

print("NB:", len(results["documents"]))

for i, (doc, meta) in enumerate(zip(results["documents"], results["metadatas"])):
    print(f"\n--- DOC {i} ---")
    print("META:", meta)
    print("TEXT:", doc)

NB: 2

--- DOC 0 ---
META: {'file_origin': 'files/Fabrication/Nourriture.txt', 'source': 'https://minecraft.fandom.com/fr/wiki/Fabrication/Nourriture', 'section': 'Introduction', 'doc_id': 'd014ee10-9ad3-4b71-a223-016827441e0f'}
TEXT: Page : Nourriture

Section : Introduction

Résumé :
Canne à sucre
Fiole de miel
Seau de lait
Champignon rouge
Champignon brun
Blé
Lingot d'or
Fèves de cacao
Citrouille
Tranche de pastèque
Pépite d'or
Œuf

--- DOC 1 ---
META: {'source': 'https://minecraft.fandom.com/fr/wiki/Fabrication/Nourriture', 'section': 'Introduction', 'doc_id': '3e35248b-3f52-4993-9a97-777d1da2bc87', 'file_origin': 'files/Fabrication/Nourriture.txt'}
TEXT: Page : Nourriture

Section : Introduction

Résumé :
- Citrouille + Œuf + Sucre : Citrouille Œuf Sucre Tarte à la citrouille
- Betterave + Bol : Betterave Betterave Betterave Betterave Betterave Betterave Bol Soupe de betteraves
- Lapin cuit + Carotte + Pomme de terre cuite + Champignon rouge ou Champignon brun + Bol : Lapin cuit C

## Retriever

#### Create a retriever using Chroma

In [24]:
print(retriever.invoke("Minecraft")[0].page_content[:500])

def retrieve_documents(retriever, query):
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if hasattr(retriever, "get_relevant_documents"):
        return retriever.get_relevant_documents(query)
    raise AttributeError("Le retriever ne supporte ni invoke ni get_relevant_documents.")

Des torches vous seront aussi utiles, non pour illuminer le Nether (il est déjà bien assez éclairé tout seul), mais pour repérer votre chemin ou pour marquer un endroit précis. Au lieu d'utiliser des torches, vous pouvez utiliser de petites colonnes avec un matériau très visible dans e Nether ( pierre taillée , sable ...). Emportez donc un stock important de ce matériau.
Enfin, n'oubliez pas d'emporter un établi pour fabriquer des armes , faire de la soupe de champignons ...
Une dernière techniq


## Generator

#### Create prompt templates

In [25]:
# Prompt template （more strict） to query Gemini
llm_prompt_template = """Tu es un assistant expert sur le jeu Minecraft.
Réponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.
Si la réponse ne se trouve pas dans le contexte ou si tu n'es pas sûr, n'invente rien. Dis EXACTEMENT : "Je suis désolé, mais l'information n'est pas dans les documents fournis."
Sois concis (5 phrases maximum) et cite la ou les sources [SOURCE_TYPE: source] à la fin de ta réponse si tu as trouvé l'information.

Question: {question}
Contexte: {context}
Réponse:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template='Tu es un assistant expert sur le jeu Minecraft.\nRéponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.\nSi la réponse ne se trouve pas dans le contexte ou si tu n\'es pas sûr, n\'invente rien. Dis EXACTEMENT : "Je suis désolé, mais l\'information n\'est pas dans les documents fournis."\nSois concis (5 phrases maximum) et cite la ou les sources [SOURCE_TYPE: source] à la fin de ta réponse si tu as trouvé l\'information.\n\nQuestion: {question}\nContexte: {context}\nRéponse:'


#### Create a stuff documents chain

In [26]:
# Combine data from documents to readable string format.
def format_docs(docs):
    formatted_docs = []

    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        content = doc.page_content

        formatted_text = f"[{source}]\n{content}"

        formatted_docs.append(formatted_text)

    return "\n\n".join(formatted_docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

### RAG AVANCÉ : ACTIVE RETRIEVAL (RRR)

In [27]:
def check_if_answered(reponse):
    prompt = f"La réponse suivante indique-t-elle qu'elle ne trouve pas l'information ou qu'elle ne sait pas ? Réponds OUI ou NON.\nRéponse : {reponse}"
    result = llm.invoke(prompt)
    text_content = str(result.content)
    return "OUI" in text_content.strip().upper()

def ask_with_active_retrieval(question):
    print(f"▶ Question posée : {question}")
    
    reponse = rag_chain.invoke(question)
    phrase_echec = "Je suis désolé, mais l'information n'est pas dans les documents fournis." # Phrase exite dans llm_prompt_template，à détecter pour déclencher l'auto-correction
    
    # Auto-correction
    if check_if_answered(reponse):
        print("Information introuvable. Activation de l'Active Retrieval...")
        
        # 1. IA de réécriture (Query Optimizer)
        llm_rewrite = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
        rewrite_prompt = f"""Tu es un expert Minecraft. Un utilisateur a posé cette question : '{question}'. 
        Cette question n'a donné aucun résultat exact dans notre base de données RAG. 
        Réécris cette question sous forme de 2 ou 3 mots-clés très simples et percutants pour optimiser une recherche par similarité vectorielle. 
        Ne renvoie QUE les mots-clés (par exemple : 'Ender Dragon stratégie'), rien d'autre."""
        
        # Générer la nouvelle requête
        nouvelle_requete = str(llm_rewrite.invoke(rewrite_prompt).content)
        print(f"Nouvelle requête optimisée par l'IA : {nouvelle_requete}")
        
        # 2. Relancer la recherche avec les nouveaux mots-clés
        reponse_niveau_2 = rag_chain.invoke(nouvelle_requete)
        
        # 3. Vérifier si la deuxième tentative a réussi
        if phrase_echec not in reponse_niveau_2:
            print("Succès du Niveau 2.")
            return f"*(Auto-correction RRR : Recherche élargie avec les mots-clés '{nouvelle_requete}')*\n\n{reponse_niveau_2}"
        else:
            print("Échec du Niveau 2. L'information n'existe définitivement pas dans la base.")
            return reponse # Renvoie le message de refus standard
            
    # Si le niveau 1 a marché directement
    print("Succès du Niveau 1.")
    return reponse

### Prompt the model

In [28]:
Markdown(ask_with_active_retrieval("Quel est l'ingrédient de base indispensable pour l'alchimie ?"))

▶ Question posée : Quel est l'ingrédient de base indispensable pour l'alchimie ?


Succès du Niveau 1.


L'ingrédient de base indispensable pour l'alchimie est la Fiole d'eau.

La Fiole d'eau est mentionnée comme étant "la base de toutes les potions" et peut être obtenue en remplissant une fiole avec l'eau d'un chaudron, d'une source d'eau ou en pêchant. [SOURCE_TYPE: minecraft.fandom.com/fr/wiki/Alchimie]

Je n'ai trouvé aucune autre information sur un ingrédient de base indispensable pour l'alchimie qui ne soit pas la Fiole d'eau.

[ SOURCE_TYPE: minecraft.fandom.com/fr/wiki/Alchimie ]

In [29]:
Markdown(ask_with_active_retrieval("C'est quoi le Warden dans Minecraft ?"))

▶ Question posée : C'est quoi le Warden dans Minecraft ?
Information introuvable. Activation de l'Active Retrieval...
Nouvelle requête optimisée par l'IA : [{'type': 'text', 'text': 'Warden Minecraft créature', 'extras': {'signature': 'Et4TCtsTAQw51sf0v+2irgzrMDn3CJMMxYrx/71HhsLW6l+hmKUcnur1fJPnNhtJGS3BcCgZs5gNEpR8+J4mk/qOpCM1d1Emc+9dx+o8k14H/imyec8z9mOOClkAyS003izYFiTbZ/tyim5Og+fZjeus+0tq1WanGbynsyRfMFaI6Ug3zquZ5ONMk3pKjRLv9dxILHThdk1UaCLQvP8Pe5UswguokTfWk2xfSBCHFmW+YeETz6cN75s+VKx0qeE24gbIKxsKv0w54BSfqgfdomAkVTAQwCuJRdiyYXp5/sHG8rA46KrLMa/8jq+QUpQ3uAmQG4bdSh9ayA9hOY6P7/xaM0v5a/lA7Op6Tdly8wFcLQZnskKpm8iV8aNY3qbVSheovJ1y2LvaxrQ8N4BD/UvIw5bym64xVhw0vo3ubiDp9/2ABgSlIssrqF1l5CMkX0t3YFrvgTVqY0ITlgAmar7V073Ye9A35iq+hkGtq0YE7GKlxnIyIlqRhMsd6+eFmT2Dyct+6FUyl3KiAw+1W9oJ99BN4dZ119emoSJPAgT6v045evMx2qXh+uVf/Sbl1FyATiWXpxKgK29Wz+uxoVJNyluLzAozHHetsyrXjKOaAI6VlCK17Ba50fk4J+AaBOLuke1fmd8OJApc3RtZNAZWnDGBuhf/pDYa3JPoL8sGy9b+gZnH7myCr9JHxA6RP3H9yywEcF3+RhNBOmHGaIrIiRmwEGkFt7V8BpK8L5F0BvMDmRafge2oUDtS

*(Auto-correction RRR : Recherche élargie avec les mots-clés '[{'type': 'text', 'text': 'Warden Minecraft créature', 'extras': {'signature': 'Et4TCtsTAQw51sf0v+2irgzrMDn3CJMMxYrx/71HhsLW6l+hmKUcnur1fJPnNhtJGS3BcCgZs5gNEpR8+J4mk/qOpCM1d1Emc+9dx+o8k14H/imyec8z9mOOClkAyS003izYFiTbZ/tyim5Og+fZjeus+0tq1WanGbynsyRfMFaI6Ug3zquZ5ONMk3pKjRLv9dxILHThdk1UaCLQvP8Pe5UswguokTfWk2xfSBCHFmW+YeETz6cN75s+VKx0qeE24gbIKxsKv0w54BSfqgfdomAkVTAQwCuJRdiyYXp5/sHG8rA46KrLMa/8jq+QUpQ3uAmQG4bdSh9ayA9hOY6P7/xaM0v5a/lA7Op6Tdly8wFcLQZnskKpm8iV8aNY3qbVSheovJ1y2LvaxrQ8N4BD/UvIw5bym64xVhw0vo3ubiDp9/2ABgSlIssrqF1l5CMkX0t3YFrvgTVqY0ITlgAmar7V073Ye9A35iq+hkGtq0YE7GKlxnIyIlqRhMsd6+eFmT2Dyct+6FUyl3KiAw+1W9oJ99BN4dZ119emoSJPAgT6v045evMx2qXh+uVf/Sbl1FyATiWXpxKgK29Wz+uxoVJNyluLzAozHHetsyrXjKOaAI6VlCK17Ba50fk4J+AaBOLuke1fmd8OJApc3RtZNAZWnDGBuhf/pDYa3JPoL8sGy9b+gZnH7myCr9JHxA6RP3H9yywEcF3+RhNBOmHGaIrIiRmwEGkFt7V8BpK8L5F0BvMDmRafge2oUDtSXMm/GhuKA9vC/I+GGh1jXe65qU1453NeNc2g0nVZ+dWoYfsbsowecXzi+TqUH6eoQJJCJlqxSpm64hNVY2DR1pLll3oxm1f4x4fLSHc3CShQa7KIHboLejQ0LXn9yIX4orChOCBeSgUWF82GpAYrMiYA+XYorWSpSbaHnoJqTqumDuV6E0fvpRP66NaEuYsMC29K74ohsL+RuyEjMqVmQjJLrKR5AKWVkN58pKMIIa94cx5pgjmhKYu0u9qE3/NDIW3RLXXmSqp2IxjcuHC9N3aUYwW4hTW4z9d0x/oUUWVwk0pviLM/efF9MilNz5yDe4BoDnQReUn37lQnum2WGHrg+iZH3DxJ2wQCSDMFvyX/Rhm/PLs/teHFnS/hu/HT0+bAfeoAEteCwtdQlz7Ikm3cSX5p5yxtqxUZ2mDLyas8PREKy0CdT9e8vaXGreCVTVfWZxNwh5ekICdvEhiFj3zcJXEo+HbpjP9Al5QIfurNfR2SGH6kfjeZRAqRlFT41P3TjxlTKZ+auvu458Owmp+/WIJU/FLhHZ9A6ewZS5d6J/hnm3N3FEJqLbAY54jH/s6oFio/8+5avqXBFYQzhmzfRxPAoqxyBsp/zWi9CfFyMqj6uhjLPEX6u0OL2cnYvLdC+kYsKLM76PBSEPEhN1YMJMsMjGXQAQ+kRREk/NN8FrSn7yTr2AE4usOKZKOZE0NJ5I4+8ZXuLBETi9rmqVDfYLSB46vXLkl4IhrhoSKfWL8fw86pa0BNoYgeUxxGtVs774JWladyHKVx5b1VaenfeD2QCPb1TPprObbw54gb5pmhjI3EJBMAed0uK2IMp4C0NVY6SxjaOLA/zglvohot/StJYA0NNPr49t4dg7M/YrujITsT2RbMLTGoy7FfdsgivVv7d8b6KKrX2n/rCKrLYTs/g+MuTOLJtxtk5ZafWirZ1szp8HbnNcpKXfe2T40OltNKq1tmL9QtRIxgWKVQw+lurOrA7l0S9vOLANPM8ord81kY/nTkfAKmpc4MHQRtIU/YFxYr06qGSJc7PvnQmarmVTYVvqBpYNrOP+7tJKZUhaniElTsoiIjAO3Xgqu1Uz05Yd3yxI87gkfv16jqkg+z5XMfwvIwHYQbMcPsKgIeuWxBFyZArNQGiVl0dTTmxEB/KZuW4QEatZGiM3GdA5/7+yEBzQz7J1aZYcl2i/L/q1FuRFSHZpYMxKW5euYM1y7/I6aJ8DRqHU5zB2KjYQVh+52+xehkM2KB6R+QMkaPX0t+ZtftxGDq1n/iSWrv9Il8ecxIgGWhLvV9DrHm3o2VOH3tqi85Xv4A7JJY4N28d79ozOkbzABlJIbirTopxqHyIGSHnVvBBxmBKe/bCQpG8XZdOK6D7Yy85dApswf35891Ee5+3vxW8mQF3oTEgMXrZqVMidwKtK5pYNRMnpyPkWu4siE9UPikeL9LOM/1UyMWGafg8iyTkiJg4kPMVieljORQx6RCvgVGmK/iUl6cymz9+04+qvMmdATcsmf9YRkKoDvsU+gK2yABxp+REi4kdMSWzkqD75Gl3/odu4KyBjK70ZQi2o6+hZ3DCfaCXc5EW9iF9VAK6ktGTraPLkSQy2RdW/8/xkcYvMWTKv0lZKAt9BrqvfNSWdp/pVd3le5xkrEWuejFOCDqIglAFlJivVIqv+qhLguRSVOeJ2yqLj+JFLzvs+GKMtTO2/cirhB+Cd3dsmTHiVYZX0avirpGeQcuW5ebr4Q0IollSBkkPiggEFOzCO+ughJ8I/kt9znnkfVMYTCY5YA0i2WZ+qpyasJvMhqUmRvj59/mTr91ff+Mrzi6+HdI+vMlq6EXbXPT+i3PGeNebgGqxI0BJCI4DJvLZNFgbftrULa1UNdEWvrP/VCYUY1FFsXmj3DVvwoC1Svog1BWF4auRxQRdF7omN5hmYYhXg01fWZ7d0YvT5dkNjSqorrbhxtKglx86zK2cAS3GbUPZzq0J4Fh6x3duwRs3fpIT8xiZObCgXILv45Gg6x3gliG6eioROuFKwfm2tu+LYcqkT7bkr2IBtVaE4XR/977CS6DtukoLbTFOtT7LxhX0wYaWUPjhAmJtcb+sX/BJZhbSbY6PWPokgiX4pzfD37l6H7as46dRacF+dHbujz7/ED7vwu6VyqVWFbWQz5rU+xyjNTWU/fmfgiEuE5w8VteUZa2IRRZxLuSjzL6dwqxh0WTKqzmaR5lHZZMO/UPdMVWiDeQznlA6h2rFwfvCgW5zDCpK9u2fslqxweT2GwusyNX+zDn+lptLCMRgruPnhZD9hJKDO/8wC9qWM/4CqtPpunZ7kXWmToKgNqJg56qQDgYBxO6AnHsP2u3xYpZYwRtr65uov4UN729dzZPYL9ldds//TDbiDs2UW/EEjEGcEbW3Ek/9+QaDjFogaF60DH7iwfW3DVzNnUWTFwWh+UWdMdAScP75wz9mRlPfRaS5j95bj9nEXPZvSkqAMbdQL9DlYXB6oC3L1H+prShLZhnbh+r1qTM1s6u6el/ddtUGpgZASQ5S/sra3mfxkPZ8BPsUu9ZvpE0+3CGlgkVqkICPjLbICEyQOAT5lkZxW4isYeQZUyZYCzY'}}]')*

Je n'ai trouvé aucune information sur le Warden dans le contexte fourni. Le Warden est une créature de Minecraft qui a été ajoutée dans la mise à jour 1.19.0.20 en 2022, mais je n'ai pas trouvé d'informations spécifiques sur son comportement ou ses attaques dans les sources fournies.

Cependant, selon les informations disponibles, le Warden est un monstre qui vit dans une "cité antique" située dans les grottes les plus basses du monde et peut infliger de très lourds dégâts. Il donne à sa mort un bloc appelé "catalyseur de sculk", également obtenable avec un outil enchanté de "toucher de soie".

In [30]:
Markdown(ask_with_active_retrieval("Comment survivre dans le Nether dans Minecraft ?"))

▶ Question posée : Comment survivre dans le Nether dans Minecraft ?
Succès du Niveau 1.


Pour survivre dans le Nether, il est essentiel d'avoir un abri sécurisé pour se protéger des dangers qui y règnent. Construire votre abri à proximité du portail et en utilisant des blocs faciles à repérer comme la pierre taillée ou le sable peut vous aider à retrouver votre chemin. Il est également important de ne pas oublier d'emporter un établi pour fabriquer des armes et faire de la soupe de champignons, ainsi que de connaître les techniques de défense contre les Ghasts.

[https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether]

Je n'ai trouvé aucune information supplémentaire sur comment survivre dans le Nether.

In [31]:
Markdown(ask_with_active_retrieval("C'est quoi l'alchimie dans Minecraft?"))

▶ Question posée : C'est quoi l'alchimie dans Minecraft?
Succès du Niveau 1.


L'alchimie dans Minecraft est la science permettant de créer des potions, des potions jetables et des potions persistantes. Elle consiste à fabriquer un alambic, le poser sur le sol et à distiller des ingrédients pour obtenir les potions désirées.

Pour commencer l'alchimie, il faut choisir un ingrédient pour l'emplacement du haut de l'alambic et atterrir des éléments compatibles dans les emplacements du bas. Le processus d'alchimie démarre lorsque les emplacements sont occupés et se termine lorsque le processus est complet.

Il est important de noter que l'alchimie peut être interrompue si l'ingrédient ou les fioles sont retirés, mais elle peut reprendre si les ingrédients sont remis.

[SOURCE_TYPE: source] [https://minecraft.fandom.com/fr/wiki/Alchimie]

In [32]:
Markdown(ask_with_active_retrieval("Il y a combien de dimensions dans Minecraft? Décris les."))

▶ Question posée : Il y a combien de dimensions dans Minecraft? Décris les.
Succès du Niveau 1.


Il y a 3 dimensions dans Minecraft : 

La Surface, le Nether et l'Endroits. La première chose à faire est donc de vous assurer que vous pourrez vous faire un abri soit en le bâtissant, soit en le creusant. Il est conseillé d'avoir un portail au cas où votre abri se ferait atteindre par une boule de feu.

[https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether]
[https://minecraft.fandom.com/fr/wiki/Cuisson]
[https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions]

In [33]:
Markdown(ask_with_active_retrieval("Parle moi des boss dans Minecraft, et décris les tous."))

▶ Question posée : Parle moi des boss dans Minecraft, et décris les tous.
Succès du Niveau 1.


Il y a plusieurs bosses dans Minecraft :

- L' Ender Dragon : un monstre puissant qui se trouve à l'intérieur de la dimension End.
- Les endermen : des créatures mystérieuses qui peuvent être utilisées pour créer des portails vers d'autres dimensions.

Ces informations ne sont pas spécifiquement mentionnées dans les documents fournis, mais elles sont généralement connues sur Minecraft. 

[SOURCE_TYPE: source]

In [34]:
### ÉVALUATION DU RAG (LLM-as-a-Judge)